In [1]:
!pip install -U datasets transformers accelerate peft trl bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from datasets import load_dataset
import json

DATASET_NAME = "minhxthanh/Vietnam-History-1M-Vi"

ds = load_dataset(DATASET_NAME, split="train")

print(ds)
print("Số mẫu:", len(ds))
print("Columns:", ds.column_names)

for i in range(3):
    print("=" * 100)
    print("ROW:", i)
    print(json.dumps(ds[i], ensure_ascii=False, indent=2)[:5000])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/628 [00:00<?, ?B/s]

Vietnam History Complete.jsonl:   0%|          | 0.00/871M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 1000000
})
Số mẫu: 1000000
Columns: ['messages']
ROW: 0
{
  "messages": [
    {
      "role": "system",
      "content": "Bạn là một trợ lý AI am hiểu về lịch sử Việt Nam. Trả lời bằng tiếng Việt, chính xác, súc tích.",
      "channel": null
    },
    {
      "role": "user",
      "content": "Tóm tắt Hòa ước Patenôtre (Giáp Thân) và nêu ý nghĩa lịch sử của sự kiện này.",
      "channel": null
    },
    {
      "role": "assistant",
      "content": "Hòa ước Patenôtre (Giáp Thân) (1884). Triều Nguyễn ký hiệp ước đặt Việt Nam dưới chế độ bảo hộ của Pháp. Chính thức hóa ách cai trị của Pháp trên toàn cõi Việt Nam.",
      "channel": "final"
    }
  ]
}
ROW: 1
{
  "messages": [
    {
      "role": "system",
      "content": "Bạn là trợ lý lịch sử Việt Nam. Hãy giải thích ngắn gọn, ưu tiên mốc thời gian và ý nghĩa.",
      "channel": null
    },
    {
      "role": "user",
      "content": "Hãy cho biết sự kiện tiêu biểu của Việt Nam vào 

In [3]:
from collections import Counter
import numpy as np

role_counter = Counter()
num_messages = []
bad_rows = 0

N = min(10000, len(ds))

for ex in ds.select(range(N)):
    msgs = ex.get("messages", None)

    if not isinstance(msgs, list):
        bad_rows += 1
        continue

    num_messages.append(len(msgs))

    for m in msgs:
        role = m.get("role", "MISSING")
        role_counter[role] += 1

print("Checked rows:", N)
print("Bad rows:", bad_rows)
print("Role counter:", role_counter)

if num_messages:
    print("Avg num messages:", np.mean(num_messages))
    print("Min num messages:", min(num_messages))
    print("Max num messages:", max(num_messages))

Checked rows: 10000
Bad rows: 0
Role counter: Counter({'assistant': 17858, 'system': 10000, 'user': 10000})
Avg num messages: 3.7858
Min num messages: 3
Max num messages: 4


In [4]:
from datasets import load_dataset
from collections import Counter
import json
import numpy as np

DATASET_NAME = "minhxthanh/Vietnam-History-1M-Vi"

ds = load_dataset(DATASET_NAME, split="train")

DEFAULT_SYSTEM_PROMPT = (
    "Bạn là trợ lý AI chuyên về Lịch sử Việt Nam. "
    "Hãy trả lời chính xác, rõ ràng, súc tích và ưu tiên mốc thời gian, nhân vật, sự kiện, ý nghĩa lịch sử."
)

def convert_final_only(example):
    messages = example["messages"]

    system_content = None
    user_content = None
    final_content = None

    for m in messages:
        role = m.get("role")
        content = (m.get("content") or "").strip()
        channel = m.get("channel")

        if not content:
            continue

        if role == "system" and system_content is None:
            system_content = content

        elif role == "user" and user_content is None:
            user_content = content

        elif role == "assistant" and channel == "final":
            final_content = content

    if system_content is None:
        system_content = DEFAULT_SYSTEM_PROMPT

    if user_content is None:
        return {
            "messages": None,
            "valid": False,
            "reason": "missing_user"
        }

    if final_content is None:
        return {
            "messages": None,
            "valid": False,
            "reason": "missing_final"
        }

    if len(final_content) < 20:
        return {
            "messages": None,
            "valid": False,
            "reason": "final_too_short"
        }

    return {
        "messages": [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": final_content}
        ],
        "valid": True,
        "reason": "ok"
    }

In [5]:
raw_small = ds.select(range(20000))

converted = raw_small.map(convert_final_only)

print(Counter(converted["reason"]))

converted = converted.filter(lambda x: x["valid"])
converted = converted.remove_columns(
    [c for c in converted.column_names if c not in ["messages"]]
)

print(converted)
print(json.dumps(converted[0], ensure_ascii=False, indent=2))

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Counter({'ok': 20000})


Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 20000
})
{
  "messages": [
    {
      "role": "system",
      "content": "Bạn là trợ lý AI chuyên về Lịch sử Việt Nam. Hãy trả lời chính xác, rõ ràng, súc tích và ưu tiên mốc thời gian, nhân vật, sự kiện, ý nghĩa lịch sử.",
      "channel": null
    },
    {
      "role": "user",
      "content": "Tóm tắt Hòa ước Patenôtre (Giáp Thân) và nêu ý nghĩa lịch sử của sự kiện này.",
      "channel": null
    },
    {
      "role": "assistant",
      "content": "Hòa ước Patenôtre (Giáp Thân) (1884). Triều Nguyễn ký hiệp ước đặt Việt Nam dưới chế độ bảo hộ của Pháp. Chính thức hóa ách cai trị của Pháp trên toàn cõi Việt Nam.",
      "channel": null
    }
  ]
}


In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def count_tokens(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {
        "num_tokens": len(tokenizer(text, add_special_tokens=False)["input_ids"])
    }

tokenized_stats = converted.select(range(min(5000, len(converted)))).map(count_tokens)

lengths = tokenized_stats["num_tokens"]

print("Num samples:", len(lengths))
print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("P90:", np.percentile(lengths, 90))
print("P95:", np.percentile(lengths, 95))
print("P99:", np.percentile(lengths, 99))
print("Max:", max(lengths))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Num samples: 5000
Mean: 139.302
Median: 137.0
P90: 158.0
P95: 164.0
P99: 176.0
Max: 189


In [7]:
MAX_LEN = 1024

def add_len_and_filter(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    n_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return {
        "num_tokens": n_tokens,
        "keep": n_tokens <= MAX_LEN
    }

converted_len = converted.map(add_len_and_filter)
print(Counter(converted_len["keep"]))

converted_clean = converted_len.filter(lambda x: x["keep"])
converted_clean = converted_clean.remove_columns(["num_tokens", "keep"])

print(converted_clean)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Counter({True: 20000})


Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 20000
})


In [8]:
converted_clean = converted_clean.shuffle(seed=42)

split = converted_clean.train_test_split(test_size=0.02, seed=42)

train_ds = split["train"]
eval_ds = split["test"]

train_ds = train_ds.select(range(min(10000, len(train_ds))))
eval_ds = eval_ds.select(range(min(300, len(eval_ds))))

print(train_ds)
print(eval_ds)
print(json.dumps(train_ds[0], ensure_ascii=False, indent=2))

Dataset({
    features: ['messages'],
    num_rows: 10000
})
Dataset({
    features: ['messages'],
    num_rows: 300
})
{
  "messages": [
    {
      "role": "system",
      "content": "Bạn là trợ lý AI chuyên về Lịch sử Việt Nam. Hãy trả lời chính xác, rõ ràng, súc tích và ưu tiên mốc thời gian, nhân vật, sự kiện, ý nghĩa lịch sử.",
      "channel": null
    },
    {
      "role": "user",
      "content": "Hậu quả hay tác động dài hạn của Nguyễn Ánh lên ngôi Gia Long, lập nhà Nguyễn đối với Việt Nam là gì?",
      "channel": null
    },
    {
      "role": "assistant",
      "content": "Vào năm 1802, diễn ra **Nguyễn Ánh lên ngôi Gia Long, lập nhà Nguyễn**: Gia Long thống nhất đất nước sau khi đánh bại Tây Sơn. Ý nghĩa: Mở đầu triều Nguyễn – triều đại phong kiến cuối cùng.",
      "channel": null
    }
  ]
}


In [9]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/qwen2_5_1_5b_vietnam_history_lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.config.use_cache = False

# BẮT BUỘC khi train QLoRA / 4-bit
model = prepare_model_for_kbit_training(model)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

In [11]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,
    num_train_epochs=3,

    max_length=1024,
    packing=False,

    logging_steps=10,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=2,

    fp16=False,
    bf16=False,
    fp16_full_eval=False,
    bf16_full_eval=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    optim="paged_adamw_8bit",

    report_to="none",
    assistant_only_loss=True,
)

In [12]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("fp16:", training_args.fp16)
print("bf16:", training_args.bf16)

trainable_dtypes = {}
for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        trainable_dtypes[str(p.dtype)] = trainable_dtypes.get(str(p.dtype), 0) + 1

print("Trainable parameter dtypes:", trainable_dtypes)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


fp16: False
bf16: False
Trainable parameter dtypes: {'torch.bfloat16': 392}


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.047852,0.045590,0.059655,446308.000000,0.982352
400,0.038096,0.035955,0.042859,892570.000000,0.985214
600,0.035422,0.034260,0.044253,1336770.000000,0.985819
800,0.034953,0.033200,0.038288,1781238.000000,0.986449
1000,0.033783,0.032974,0.038370,2227352.000000,0.985452
1200,0.031338,0.032885,0.034686,2672334.000000,0.986428
1400,0.031682,0.032183,0.036800,3119165.000000,0.985812


KeyboardInterrupt: 

In [15]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [17]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Đổi checkpoint ở đây
ADAPTER_DIR = "/content/qwen2_5_1_5b_vietnam_history_lora/checkpoint-1200"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model.eval()

SYSTEM_PROMPT = (
    "Bạn là trợ lý am hiểu lịch sử Việt Nam. "
    "Trả lời chính xác, ngắn gọn, dễ hiểu. "
    "Nếu câu hỏi yêu cầu so sánh hoặc giải thích, hãy trình bày theo ý chính rõ ràng."
)

test_questions = [
    "Hãy giải thích ngắn gọn nguyên nhân thắng lợi của cuộc kháng chiến chống quân Nguyên Mông.",
    "Vì sao chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử quan trọng?",
    "Vai trò của Nguyễn Trãi trong khởi nghĩa Lam Sơn là gì?",
    "Hãy nêu các cải cách tiêu biểu của Hồ Quý Ly.",
    "So sánh chính sách cai trị của nhà Lý và nhà Trần.",
    "Nguyên nhân nào dẫn đến sự suy yếu của triều Nguyễn cuối thế kỷ XIX?",
    "Phong trào Cần Vương có ý nghĩa gì trong lịch sử Việt Nam?",
    "Vì sao cuộc khởi nghĩa Hai Bà Trưng được xem là biểu tượng của tinh thần độc lập dân tộc?",
    "Hãy giải thích ngắn gọn nguyên nhân thắng lợi của Cách mạng tháng Tám năm 1945.",
    "Nêu ý nghĩa lịch sử của chiến dịch Điện Biên Phủ năm 1954."
]

def generate_answer(question, max_new_tokens=300):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()


print("Loaded adapter from:", ADAPTER_DIR)
print("Tokenizer loaded from:", BASE_MODEL)
print()

for i, question in enumerate(test_questions, start=1):
    print("=" * 100)
    print(f"Câu {i}: {question}")
    print("-" * 100)

    answer = generate_answer(question)
    print(answer)
    print()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded adapter from: /content/qwen2_5_1_5b_vietnam_history_lora/checkpoint-1200
Tokenizer loaded from: Qwen/Qwen2.5-1.5B-Instruct

Câu 1: Hãy giải thích ngắn gọn nguyên nhân thắng lợi của cuộc kháng chiến chống quân Nguyên Mông.
----------------------------------------------------------------------------------------------------
Vào năm 1285, diễn ra **Cuộc kháng chiến chống quân Nguyên Mông**: Quân Đại Việt đánh bại quân Nguyên Mông trên nhiều phương diện. Ý nghĩa: Bảo vệ độc lập, khẳng định vị thế quốc gia.

Câu 2: Vì sao chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử quan trọng?
----------------------------------------------------------------------------------------------------
Chiến thắng Bạch Đằng (938): Quân Đại Việt đánh bại thủy quân Nam Hán trên sông Bạch Đằng. Bảo vệ bờ cõi, củng cố nền tự chủ.

Câu 3: Vai trò của Nguyễn Trãi trong khởi nghĩa Lam Sơn là gì?
----------------------------------------------------------------------------------------------------
Nguyễn Trãi khởi n

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()

SYSTEM_PROMPT = (
    "Bạn là trợ lý am hiểu lịch sử Việt Nam. "
    "Trả lời chính xác, ngắn gọn, dễ hiểu. "
    "Nếu câu hỏi yêu cầu so sánh hoặc giải thích, hãy trình bày theo ý chính rõ ràng."
)

test_questions = [
    "Hãy giải thích ngắn gọn nguyên nhân thắng lợi của cuộc kháng chiến chống quân Nguyên Mông.",
    "Vì sao chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử quan trọng?",
    "Vai trò của Nguyễn Trãi trong khởi nghĩa Lam Sơn là gì?",
    "Hãy nêu các cải cách tiêu biểu của Hồ Quý Ly.",
    "So sánh chính sách cai trị của nhà Lý và nhà Trần.",
    "Nguyên nhân nào dẫn đến sự suy yếu của triều Nguyễn cuối thế kỷ XIX?",
    "Phong trào Cần Vương có ý nghĩa gì trong lịch sử Việt Nam?",
    "Vì sao cuộc khởi nghĩa Hai Bà Trưng được xem là biểu tượng của tinh thần độc lập dân tộc?",
    "Hãy giải thích ngắn gọn nguyên nhân thắng lợi của Cách mạng tháng Tám năm 1945.",
    "Nêu ý nghĩa lịch sử của chiến dịch Điện Biên Phủ năm 1954."
]

def generate_answer(question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

for i, question in enumerate(test_questions, start=1):
    print("=" * 100)
    print(f"Câu {i}: {question}")
    print("-" * 100)
    print(generate_answer(question))
    print()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Câu 1: Hãy giải thích ngắn gọn nguyên nhân thắng lợi của cuộc kháng chiến chống quân Nguyên Mông.
----------------------------------------------------------------------------------------------------
Cuộc kháng chiến chống quân Nguyên Mông của Việt Nam có nhiều nguyên nhân chính:

1. Tinh thần yêu nước và lòng tự tôn dân tộc của người Việt:
- Người Việt luôn coi mình là dân tộc độc lập, không chịu khuất phục trước bất kỳ kẻ thù nào.
- Có truyền thống đấu tranh chống ngoại xâm từ thời cổ đại.

2. Sự lãnh đạo của các领袖 nổi tiếng như Nguyễn Trãi, Lê Lợi:
- Họ đã đưa ra những chiến lược và kế hoạch tinh vi để đánh bại quân Nguyên Mông.
- Có sự phối hợp chặt chẽ giữa quân đội và dân chúng.

3. Sự hỗ trợ của nhân dân:
- Người dân đã tham gia tích cực vào cuộc chiến đấu.
- Có sự ủng hộ mạnh mẽ từ các nước bạn như Trung Quốc, Nhật Bản.

4. Kỹ năng chiến thuật của các领袖:
- Nguyễn Trãi đã sáng tạo ra chiến lược "tổng tiến" để đánh bại quân Nguyên Mông.
- Lê Lợi đã sử dụng chiến thuật "đầu hàng" đ

In [19]:
import random

for i in random.sample(range(len(train_ds)), 20):
    ex = train_ds[i]
    print("=" * 100)
    print("Index:", i)

    if "messages" in ex:
        for m in ex["messages"]:
            print(f"\n[{m.get('role')}]")
            print(m.get("content"))
    else:
        print(ex)

Index: 1824

[system]
Bạn là trợ lý AI chuyên về Lịch sử Việt Nam. Hãy trả lời chính xác, rõ ràng, súc tích và ưu tiên mốc thời gian, nhân vật, sự kiện, ý nghĩa lịch sử.

[user]
Năm 1483 ở Việt Nam diễn ra sự kiện gì quan trọng?

[assistant]
**Bộ luật Hồng Đức** xảy ra năm 1483. Bộ luật Hồng Đức hoàn thiện dưới thời Lê Thánh Tông. Đây là cột mốc quan trọng vì Bộ luật tiêu biểu của Việt Nam thời phong kiến.
Index: 409

[system]
Bạn là trợ lý AI chuyên về Lịch sử Việt Nam. Hãy trả lời chính xác, rõ ràng, súc tích và ưu tiên mốc thời gian, nhân vật, sự kiện, ý nghĩa lịch sử.

[user]
Giải thích vì sao Đổi quốc hiệu thành Đại Việt được coi là cột mốc quan trọng.

[assistant]
**Đổi quốc hiệu thành Đại Việt** xảy ra năm 1054. Thời Lý Thánh Tông, quốc hiệu đổi từ Đại Cồ Việt sang Đại Việt. Đây là cột mốc quan trọng vì Khẳng định bản sắc và vị thế quốc gia độc lập.
Index: 4506

[system]
Bạn là trợ lý AI chuyên về Lịch sử Việt Nam. Hãy trả lời chính xác, rõ ràng, súc tích và ưu tiên mốc thời gia